# Architecture de routage synaptique (SRA)
## 11 : Suppression dynamique des synapses (Annulation du Hot-Swap/Purge d'un domaine spécifique)

Dans ce cahier, nous démontrons une fonctionnalité de SRA : la « suppression synaptique ». Plus précisément, nous examinerons les deux scénarios suivants.
1. **Supprimer le plugin (`pop_synapses`)** : Supprimez physiquement les synapses ajoutées ultérieurement (Hot-Swap) à la fin et restaurez-les à l'état où elles étaient avant leur ajout.
2. **Purgez les domaines spécifiques (`clear_synapses`)** : extrayez uniquement les fonctions spécifiques (par exemple, les mathématiques) du modèle de base initial et « zéro effacement » en toute sécurité les synapses qui ne sont pas partagées avec d'autres.

*Cet ordinateur portable peut fonctionner sans GPU.

In [ ]:
# 0. 環境セットアップ (Google Colab用)
import sys
import os

if 'google.colab' in sys.modules:
    if not os.path.exists('SynapticRouter'):
        !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install tiktoken torch

# パスの追加
sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

from sra_language_models import MoESRALanguageModel
from sra_experiment import find_unshared_synapses
import torch


### 1. Suppression des plugins (`pop_synapses`)
Tout d'abord, construisons un petit modèle, ajoutons dynamiquement des synapses, puis supprimons-les avec `pop_synapses`.

In [ ]:
# モデルの初期化
dim = 128
layers = 2
num_synapses = 4
k = 2
syn_hidden = 256
vocab_size = 1000

print("=== プラグインの取り外しデモ ===")
model = MoESRALanguageModel(vocab_size, dim, layers, num_synapses, k, syn_hidden)
print(f"[初期状態] 搭載シナプス数: {model.blocks[0].num_synapses}")
print(f"  ルーターの埋め込みテンソル形状: {model.blocks[0].router.get_full_emb().shape}")

# シナプスの追加 (Hot-Swap)
print("\nプラグインとしてシナプスを 2つ 追加します...")
model.add_synapses(2, freeze_base=True)
print(f"[追加後] 搭載シナプス数: {model.blocks[0].num_synapses}")
print(f"  ルーターの埋め込みテンソル形状: {model.blocks[0].router.get_full_emb().shape}")

# シナプスの取り外し (Undo Hot-Swap)
print("\n追加したシナプスを 1つ 取り外します...")
model.pop_synapses(1)
print(f"[取り外し後] 搭載シナプス数: {model.blocks[0].num_synapses}")
print(f"  ルーターの埋め込みテンソル形状: {model.blocks[0].router.get_full_emb().shape}")

De cette façon, l'appel de « pop_synapses(N) » supprime physiquement les N tenseurs synaptiques de la fin, restaurant ainsi la capacité du modèle.

### 2. Purge de domaines spécifiques (`clear_synapses` et `find_unshared_synapses`)
Ensuite, nous démontrerons le processus de détection automatique des « synapses inutiles qui ne sont utilisées que dans un domaine spécifique (dans ce cas, « mathématiques ») » parmi le modèle de base qui a appris plusieurs domaines, et de les désactiver en toute sécurité en les effaçant.

In [ ]:
import random
tokenizer = tiktoken.get_encoding("cl100k_base")
vocab_size = tokenizer.n_vocab

device = "cuda" if torch.cuda.is_available() else "cpu"

# 仮のデータセットとバッチ関数の準備（デモ用）
domains = ["text", "code", "math"]
data = {d: torch.randint(0, vocab_size, (1000,)) for d in domains}

def dummy_get_batch(data_dict, batch_size, seq_len, domain):
    x = torch.zeros((batch_size, seq_len), dtype=torch.long)
    y = torch.zeros((batch_size, seq_len), dtype=torch.long)
    d_data = data_dict[domain]
    for i in range(batch_size):
        start = random.randint(0, len(d_data) - seq_len - 1)
        x[i] = d_data[start:start+seq_len]
        y[i] = d_data[start+1:start+seq_len+1]
    return x, y

# もう少し大きめのモデルを準備
multi_model = MoESRALanguageModel(vocab_size, dim=128, layers=2, num_synapses=16, k=2, syn_hidden=256).to(device)

In [ ]:
print("=== 特定ドメインのパージデモ ===")
print("Mathドメインでのみ使用され、他(Text, Code)と共用されていないシナプスを検索します...")

# ユーティリティを用いて対象シナプスを特定
unshared_synapses = find_unshared_synapses(
    model=multi_model, 
    data_dict=data, 
    target_domain="math", 
    other_domains=["text", "code"], 
    get_batch_func=dummy_get_batch,
    max_seq_len=32,
    threshold=0.01  # 1%以上の頻度で利用されていれば「使用している」と判定
)

print(f"\n抽出されたMath専用のシナプスインデックス: {unshared_synapses}")

*L'indice ci-dessus peut ne pas être trouvé dans le cas d'un modèle non entraîné car il est distribué de manière aléatoire.

Transmettez l'index extrait (ou un index approprié à des fins de démonstration s'il n'est pas trouvé) à `clear_synapses` pour mettre à zéro la synapse correspondante.

In [ ]:
if len(unshared_synapses) > 0:
    target_idx = unshared_synapses[0]
else:
    print("※未学習モデルのため条件に完全合致するものがありませんでした。デモとしてシナプス 3 を対象にします。")
    unshared_synapses = [3]
    target_idx = 3

# クリア前の重みのノルムを確認
pre_emb_norm = torch.norm(multi_model.blocks[0].router.get_full_emb()[target_idx]).item()
pre_w1_norm = torch.norm(multi_model.blocks[0].get_full_param('w1')[target_idx]).item()
print(f"[クリア前] シナプス {target_idx} の Router埋め込みノルム: {pre_emb_norm:.4f}, W1ノルム: {pre_w1_norm:.4f}")

# ゼロクリア実行
multi_model.clear_synapses(unshared_synapses)
print("\nゼロクリアを実行しました。\n")

# クリア後の重みのノルムを確認
post_emb_norm = torch.norm(multi_model.blocks[0].router.get_full_emb()[target_idx]).item()
post_w1_norm = torch.norm(multi_model.blocks[0].get_full_param('w1')[target_idx]).item()
print(f"[クリア後] シナプス {target_idx} の Router埋め込みノルム: {post_emb_norm:.4f}, W1ノルム: {post_w1_norm:.4f}")

### Conclusion
`clear_synapses` efface le poids de l'index spécifié à `0.0` sans le supprimer physiquement.
Cela empêche les index (ID) des autres synapses de dériver et désactive complètement les chemins de calcul inutiles tout en conservant la compatibilité avec les masques de métadonnées existants. Il est possible d'écraser (Hot-Swap) l'index avec un nouveau plugin ultérieurement lorsqu'il devient un emplacement vide.